# Version 2 Data Validation

This notebook reviews the automated Checkpoint 37 outputs. The calculations are created by `src/validate_v2_attrition_data.py`; the notebook is a readable inspection layer rather than a second implementation.

In [ ]:
from pathlib import Path
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve().parent
OUTPUT_DIR = PROJECT_ROOT / "data" / "processed" / "v2_validation"

def read_output(filename):
    return pd.read_csv(OUTPUT_DIR / filename)

## 1. Validation status

Every required integrity, calibration, signal, hierarchy, and leakage check must pass before the project proceeds to temporal dataset construction.

In [ ]:
validation = read_output("validation_summary.csv")
validation

In [ ]:
validation["status"].value_counts()

## 2. Version 1 versus Version 2

Cumulative attrition and the twelve-month modeling target are different quantities. The comparison keeps them on separate rows.

In [ ]:
version_comparison = read_output("v1_v2_comparison.csv")
version_comparison

In [ ]:
snapshot_summary = read_output("snapshot_summary.csv")
snapshot_summary

## 3. Organizational and department outcomes

Version 2 permits team-manager exits while protecting department heads and senior managers. This removes the Version 1 artifact in which every leader was permanently active.

In [ ]:
organizational = read_output("attrition_by_organizational_level.csv")
departments = read_output("attrition_by_department.csv")
display(organizational)
display(departments)

## 4. Observable signal checks

No single feature should nearly determine the outcome. At the same time, a simple cross-validated model should detect moderate signal.

In [ ]:
numeric_signals = read_output("numeric_signal_checks.csv")
numeric_signals.head(12)

In [ ]:
categorical_signals = read_output("categorical_signal_checks.csv")
(
    categorical_signals
    .loc[categorical_signals["included_in_rate_check"]]
    .sort_values("rate_ratio_vs_baseline", ascending=False)
    .head(15)
)

In [ ]:
diagnostic_model = read_output("diagnostic_model_summary.csv")
diagnostic_folds = read_output("diagnostic_model_folds.csv")
display(diagnostic_model)
display(diagnostic_folds)

## 5. Interpretation

- The Version 2 twelve-month positive rate is inside the configured 7%–13% range.
- Voluntary and involuntary causes remain stochastic rather than deterministic.
- The strongest individual numeric correlation remains well below the 0.40 limit.
- A five-fold Logistic Regression diagnostic reaches the configured minimum ROC-AUC without producing unrealistically easy prediction.
- PR-AUC exceeds the no-skill class-prevalence baseline.
- All feature-source cutoffs are on or before the snapshot date.

The diagnostic model is only a generator-quality test. Formal temporal splitting, model comparison, calibration, and threshold selection occur in later checkpoints.